<a href="https://colab.research.google.com/github/amigli/AgilityDog/blob/master/PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

In [2]:
!git clone https://{userdata.get('TokenGithub')}"@github.com/amigli/Q-Bert_RL.git"

Cloning into 'Q-Bert_RL'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 553 (delta 16), reused 20 (delta 14), pack-reused 526 (from 1)
Receiving objects: 100% (553/553), 6.47 MiB | 5.61 MiB/s, done.
Resolving deltas: 100% (352/352), done.


In [3]:
%cd Q-Bert_RL/

/content/Q-Bert_RL


In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install wandb

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
directory_videos = '/content/Videos/'

## Algoritmo

In [6]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import ale_py
from tqdm import tqdm
import matplotlib.pyplot as plt
import tensorflow as tf
from stable_baselines3.common.logger import configure
import torch
from torch.utils.tensorboard import SummaryWriter
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv
from EnvironmentWrappers.RewardFunction import RewardFunction
from EnvironmentWrappers.ObsRewardWrapper import ObsRewardWrapper
import wandb
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

In [ ]:
# Non eseguire, c'è Wandb
tensorboard_log_dir = "./tensorboard_logs/"

## Wandb

In [7]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [8]:
# Inizializzazione di wandb
wandb.init(
    project="QBERT-RL",
    entity = "Q-BertRLTeam",

    config={
        "learning_rate": 0.0003,
        "epochs": 100,
    }
)

/usr/local/lib/python3.11/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
wandb: Currently logged in as: miglinoannalaura (Q-BertRLTeam). Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


## Training

In [9]:
gym.register_envs(ale_py)

## TRAINING
env = Monitor(gym.make("ALE/Qbert-ram-v5"))
env = ObsRewardWrapper(env)

In [10]:
model = PPO(
    "MlpPolicy",      # Tipo di rete neurale (MLP per osservazioni vettoriali)
    env,              # Ambiente
    verbose=1,        # Livello di logging
    learning_rate=0.0003,  # Tasso di apprendimento
    n_steps=2048,     # Passi raccolti per ogni aggiornamento
    batch_size=64,    # Dimensione del batch per l'ottimizzazione
    n_epochs=100,     # Numero di epoche per aggiornare il modello
    gamma=0.99        # Fattore di sconto
    )

Using cpu device
Wrapping the env in a DummyVecEnv.


In [ ]:
# Non avviare, c'è Wandb
%load_ext tensorboard
%tensorboard --logdir ./ppo_tensorboard/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 1174), started 0:00:04 ago. (Use '!kill 1174' to kill it.)

<IPython.core.display.Javascript object>

In [11]:
class WandbCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(WandbCallback, self).__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self) -> bool:
        """Viene chiamato ad ogni step, logga i dati degli episodi."""

        for info in self.locals["infos"]:
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])

        if len(self.episode_rewards) > 0:
            wandb.log({
                "rollout/ep_rew_mean": sum(self.episode_rewards) / len(self.episode_rewards),
                "rollout/ep_len_mean": sum(self.episode_lengths) / len(self.episode_lengths),
                "train/loss": self.model.logger.name_to_value.get("train/loss", 0),
                "train/policy_gradient_loss": self.model.logger.name_to_value.get("train/policy_gradient_loss", 0),
                "train/value_loss": self.model.logger.name_to_value.get("train/value_loss", 0),
            })

        return True


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [12]:
model.learn(total_timesteps = 10000, callback=WandbCallback())

wandb.finish()

# Valutazione del modello
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Ricompensa media: {mean_reward:.2f}, deviazione standard: {std_reward:.2f}")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 325      |
|    ep_rew_mean     | 138      |
| time/              |          |
|    fps             | 432      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 324         |
|    ep_rew_mean          | 121         |
| time/                   |             |
|    fps                  | 152         |
|    iterations           | 2           |
|    time_elapsed         | 26          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012526934 |
|    clip_fraction        | 0.193       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.78       |
|    explained_variance   | 0.0258      |
|    learning_rate        | 0.

rollout/ep_len_mean,▁▄▄▄▃▆▆▆▆▆████████▇▆▆▇▇▇▇▇▇▇▇▇█▇████████
rollout/ep_rew_mean,▁▁▃▃▇▇▇▇▇▇▇▇▇▇▇▆▆▆▆▆▇▇▆▆▆▆▇▇▇▇▇▇▇▇██████
train/loss,▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄███▄▄▄▄▄▄▄▄▄▄▄▄
train/policy_gradient_loss,████████▁▁▁▁▁▁▁▁▁▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂
train/value_loss,▁▁▁▁▁▁▁▁▁▁████████▇▇▇▇▇▇▇▇▇▇▇▄▄▄▄▄▄▄▄▄▄▄
rollout/ep_len_mean,329.83871
rollout/ep_rew_mean,145.96774
train/loss,0.43029
train/policy_gradient_loss,-0.01343
train/value_loss,0.95581


Ricompensa media: 0.00, deviazione standard: 0.00


In [ ]:
env = gym.make("ALE/Qbert-ram-v5", render_mode="rgb_array")
env = ObsRewardWrapper(env)

In [ ]:
# DummyVecEnv per compatibilità con Stable-Baselines3
env = DummyVecEnv([lambda: env])

In [ ]:
# Registra video
video_folder = "./videos/"
env = VecVideoRecorder(
    env,               # Ambiente
    video_folder,      # Cartella per salvare i video
    record_video_trigger=lambda x: x % 1000 == 0,  # Registra ogni 1000 passi
    video_length=1000 # Durata massima del video in passi
)

In [ ]:
# Resetta l'ambiente per registrare un episodio
obs = env.reset()

# Registra 3 episodi
for episode in range(3):
    obs = env.reset()
    for _ in range(1000):  # Durata massima dell'episodio
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, info = env.step(action)
        if dones[0]:  # L'episodio è terminato
            break

env.close()  # Salva il video

Saving video to /content/videos/rl-video-step-1000-to-step-2000.mp4
Moviepy - Building video /content/videos/rl-video-step-1000-to-step-2000.mp4.
Moviepy - Writing video /content/videos/rl-video-step-1000-to-step-2000.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/rl-video-step-1000-to-step-2000.mp4
